# Test 3: ajuste de inferencia

[Guía de la entrega](../../../../docs/README.md) · [Caso e historial](../README.md)

Se ajustan **temperatura, modo de razonamiento y límite de salida** con los pesos del modelo fijos. No hay fine-tuning, entrenamiento ni Colab. El objetivo es encontrar una configuración que resuelva la negociación sintética antes de retomar la automejora.

Este notebook reproduce los resultados guardados; ejecutarlo no llama a ningún modelo.

**Resultado:** Qwen 8B con razonamiento resolvió 12/12 casos conocidos y 24/24 ejecuciones de confirmación: ocho casos reservados, tres repeticiones, sin incumplimientos. Se conserva esta configuración para el siguiente experimento. La política es manual y fija; todavía no se ha validado la automejora con este modo.

El barrido directo de 8B/32B y los dos controles API no alcanzaron el criterio completo. Se conservan todos sus resultados.

## Antes de leer los resultados

Se ajusta la inferencia con una política manual fija. El negociador recibe deudas e historial;
el evaluador comprueba acción, deuda, pagos y cuota. **Q** es la proporción de decisiones
correctas sin incumplimientos; **V**, la proporción de respuestas con alguna infracción.
El criterio del piloto es Q ≥ 80 % y V = 0.

No se entrenan pesos ni se ejecuta el mejorador en este test. Se prueba primero temperatura;
después, un perfil con razonamiento y muestreo recomendado. Los Tests 4 y 5 retoman el bucle
completo de la figura. Los resultados guardados pueden leerse sin ejecutar las celdas.

![Bucle de mejora que se estudia después de este control de inferencia](../diagrams/learning-loop.svg)


In [1]:
from pathlib import Path
import hashlib
import html
import json
from IPython.display import HTML, display

relative = Path("research/cases/case-0003-debt-negotiation")
case_dir = next(base / relative for base in [Path.cwd(), *Path.cwd().parents]
                if (base / relative / "results").is_dir())
run_names = ['20261106T202558Z-inference-qwen3-8b-tuning', '20261106T203000Z-inference-qwen3-32b-tuning', '20261106T204139Z-inference-qwen3-8b-thinking-tuning', '20261106T204259Z-inference-qwen3-8b-thinking-confirmation']

def load_record(name):
    folder = case_dir / "results" / name
    manifest = json.loads((folder / "manifest.json").read_text())
    for filename, digest in manifest.items():
        assert Path(filename).name == filename
        assert hashlib.sha256((folder / filename).read_bytes()).hexdigest() == digest
    return {"name": name, "protocol": json.loads((folder / "protocol.json").read_text()),
            "report": json.loads((folder / "report.json").read_text())}

records = [load_record(name) for name in run_names]
def table(headers, rows):
    head = "".join(f'<th style="text-align:left;padding:8px;border-bottom:2px solid #667085">{html.escape(str(h))}</th>' for h in headers)
    body = "".join("<tr>" + "".join(f'<td style="text-align:left;vertical-align:top;overflow-wrap:anywhere;padding:8px;border-bottom:1px solid #ddd">{html.escape(str(v))}</td>' for v in row) + "</tr>" for row in rows)
    display(HTML(f'<table style="width:100%;border-collapse:collapse;margin:16px 0"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))
print(f"Integridad verificada: {len(records)} ejecuciones locales.")

def model_label(item):
    return item["protocol"]["model_key"] + (" · razonamiento" if item["protocol"].get("enable_thinking") else " · directo")

Integridad verificada: 4 ejecuciones locales.


## 1. Qué se ajusta

| Elemento | Configuración |
|---|---|
| Hiperparámetro | Temperatura: 0; 0,3; 0,7 |
| Controles | `top_p=0.8`, `top_k=20`, máximo 512 tokens de salida; razonamiento desactivado |
| Repeticiones | Temperatura 0: una; 0,3 y 0,7: semillas 11, 22 y 33 |
| Política | Misma referencia manual, sin mejorador ni actualizador |
| Ajuste | 12 escenarios conocidos, 84 llamadas por modelo |
| Confirmación | Ocho escenarios fijados antes del barrido, tres repeticiones; solo para un perfil que cumpla |

Temperatura 0 usa selección determinista: `top_p` y `top_k` no cambian esa decisión. Con muestreo, la temperatura modifica la distribución de probabilidades. El tamaño del modelo es un contraste adicional, no fine-tuning.

La ficha de [Qwen3](https://huggingface.co/Qwen/Qwen3-8B#best-practices) propone temperatura 0,7, `top_p=0.8` y `top_k=20` sin razonamiento. Se incluye ese perfil y uno intermedio.

**Extensión tras el barrido:** Qwen 8B con razonamiento activado, temperatura 0,6, `top_p=0.95`, `top_k=20`, salida hasta 2.048 tokens. Se prueba una configuración sobre los 12 casos conocidos. Solo si cumple pasa a confirmación. Cambian modo, muestreo y presupuesto de salida juntos; no se atribuye el resultado a uno solo.

## 2. Criterio antes de ejecutar

$$Q=\frac{\text{decisiones correctas sin incumplimientos}}{\text{ejecuciones}},\qquad V=\frac{\text{ejecuciones con incumplimientos}}{\text{ejecuciones}}.$$

**Criterio piloto:** $Q\geq0.80$ y $V=0$. La decisión debe coincidir en acción, deuda, pagos y cuota con lo esperado. Ejemplos de incumplimiento: exceder la capacidad, inventar una aceptación o calcular mal la cuota.

En el barrido local se priorizan perfiles sin incumplimientos; después, mayor Q, menor V y menor temperatura. Se conserva el barrido completo. La confirmación se consulta una vez seleccionada la configuración; su resultado no se utiliza para volver a ajustarla.

Superar este criterio en pocos casos sintéticos acredita esta prueba, no fiabilidad general ni automejora.

In [2]:
rows = []
for item in records:
    if item["protocol"]["phase"] != "tuning":
        continue
    for profile in item["report"].get("profiles", []):
        rows.append([model_label(item), profile["sampling"]["temperature"],
                     f'{profile["successes"]}/{profile["count"]}', f'{profile["q"]:.1%}',
                     f'{profile["v"]:.1%}', "Cumple" if profile["passes"] else "No cumple"])
table(["Modelo", "Temperatura", "Aciertos", "Q", "V", "Criterio piloto"], rows)

Modelo,Temperatura,Aciertos,Q,V,Criterio piloto
qwen3-8b · directo,0.0,6/12,50.0%,0.0%,No cumple
qwen3-8b · directo,0.3,18/36,50.0%,0.0%,No cumple
qwen3-8b · directo,0.7,18/36,50.0%,0.0%,No cumple
qwen3-32b · directo,0.0,8/12,66.7%,33.3%,No cumple
qwen3-32b · directo,0.3,24/36,66.7%,27.8%,No cumple
qwen3-32b · directo,0.7,24/36,66.7%,27.8%,No cumple
qwen3-8b · razonamiento,0.6,12/12,100.0%,0.0%,Cumple


## 3. Repeticiones y consumo

Las repeticiones reutilizan escenarios; no son casos independientes adicionales. El tiempo incluye carga del modelo, no descarga. El pico MLX no equivale a toda la RAM del equipo.

In [3]:
rows = []
for item in records:
    for result in item["report"]["evaluations"]:
        rows.append([model_label(item), item["protocol"]["phase"], result["sampling"]["temperature"],
                     result["sampling"]["seed"], f'{result["q"]:.1%}', f'{result["v"]:.1%}'])
table(["Modelo", "Etapa", "Temperatura", "Semilla", "Q", "V"], rows)
table(["Modelo", "Etapa", "Llamadas", "Tokens", "Segundos", "Pico MLX GB"],
      [[model_label(r), r["protocol"]["phase"], r["report"]["calls"], r["report"]["tokens"],
        round(r["report"]["elapsed_seconds"],1), round(r["report"]["peak_mlx_gb"] or 0,2)] for r in records])

Modelo,Etapa,Temperatura,Semilla,Q,V
qwen3-8b · directo,tuning,0.0,11,50.0%,0.0%
qwen3-8b · directo,tuning,0.3,11,50.0%,0.0%
qwen3-8b · directo,tuning,0.3,22,50.0%,0.0%
qwen3-8b · directo,tuning,0.3,33,50.0%,0.0%
qwen3-8b · directo,tuning,0.7,11,50.0%,0.0%
qwen3-8b · directo,tuning,0.7,22,50.0%,0.0%
qwen3-8b · directo,tuning,0.7,33,50.0%,0.0%
qwen3-32b · directo,tuning,0.0,11,66.7%,33.3%
qwen3-32b · directo,tuning,0.3,11,66.7%,25.0%
qwen3-32b · directo,tuning,0.3,22,66.7%,25.0%


Modelo,Etapa,Llamadas,Tokens,Segundos,Pico MLX GB
qwen3-8b · directo,tuning,84,70154,105.4,5.43
qwen3-32b · directo,tuning,84,70673,539.1,19.35
qwen3-8b · razonamiento,tuning,12,15110,65.4,5.42
qwen3-8b · razonamiento,confirmation,24,31036,178.3,5.42


## 4. Confirmación

La política y el evaluador son los mismos. Cambian los escenarios: capacidades 110, 175, 310, 650 y 90; capacidad desconocida, rechazo de negociación y aceptación explícita. Se comprueba la configuración elegida sin cambiarla después de ver estas respuestas.

In [4]:
confirmed = [r for r in records if r["protocol"]["phase"] == "confirmation"]
rows = []
for item in confirmed:
    result = item["report"]["confirmation"]
    rows.append([model_label(item), f'{sum(r["success"] for r in result["rows"])}/{len(result["rows"])}',
                 f'{result["q"]:.1%}', f'{result["v"]:.1%}', "Cumple" if item["report"]["passes"] else "No cumple"])
table(["Modelo", "Aciertos", "Q", "V", "Criterio piloto"], rows)
if not confirmed:
    print("No se llevó ningún perfil local a confirmación: ninguno cumplió el criterio de ajuste.")
for item in confirmed:
    rows = []
    for case in item["protocol"]["cases"]:
        outputs = [row for result in item["report"]["evaluations"] for row in result["rows"] if row["id"] == case["id"]]
        expected = " / ".join(str(case["expected"][k]) for k in ("action","debt","payments","installment"))
        rows.append([case["id"], expected, f'{sum(r["success"] for r in outputs)}/{len(outputs)}',
                     "; ".join(sorted({json.dumps(r["raw_reply"],ensure_ascii=False) for r in outputs}))])
    table(["Caso", "Esperado: acción/deuda/pagos/cuota", "Aciertos", "Respuestas originales"], rows)

Modelo,Aciertos,Q,V,Criterio piloto
qwen3-8b · razonamiento,24/24,100.0%,0.0%,Cumple


Caso,Esperado: acción/deuda/pagos/cuota,Aciertos,Respuestas originales
confirmation-110,offer / A / 2 / 100,3/3,"{""action"": ""offer"", ""debt"": ""A"", ""payments"": 2, ""installment"": 100}"
confirmation-175,offer / B / 2 / 150,3/3,"{""action"": ""offer"", ""debt"": ""B"", ""payments"": 2, ""installment"": 150}"
confirmation-310,offer / C / 2 / 300,3/3,"{""action"": ""offer"", ""debt"": ""C"", ""payments"": 2, ""installment"": 300}"
confirmation-650,offer / C / 1 / 600,3/3,"{""action"": ""offer"", ""debt"": ""C"", ""payments"": 1, ""installment"": 600}"
confirmation-90,stop / None / 0 / 0,3/3,"{""action"": ""stop"", ""debt"": null, ""payments"": 0, ""installment"": 0}"
confirmation-unknown,ask / None / 0 / 0,3/3,"{""action"": ""ask"", ""debt"": null, ""payments"": 0, ""installment"": 0}"
confirmation-decline,stop / None / 0 / 0,3/3,"{""action"": ""stop"", ""debt"": null, ""payments"": 0, ""installment"": 0}"
confirmation-consent,accept / A / 2 / 100,3/3,"{""action"": ""accept"", ""debt"": ""A"", ""payments"": 2, ""installment"": 100}"


## 5. Control por API: GPT-4o mini

Se conserva la política, los casos y el evaluador. Se usa el snapshot `gpt-4o-mini-2024-07-18`, con salida JSON estructurada. Se registran el muestreo predeterminado y un ajuste posterior a temperatura cero. El formato se garantiza de otra manera que en MLX; esta comparación no aísla el efecto del tamaño del modelo.

Primero se comprueban los 12 casos conocidos. Solo si cumplen el criterio se ejecutan los ocho reservados tres veces. No se ajusta el prompt a partir de estas respuestas.

In [5]:
api_records = [load_record(name) for name in ['20261106T203920Z-api-reference', '20261106T204025Z-api-reference']]
for item in api_records:
    report = item['report']
    table(['Ejecución', 'Temperatura'], [[item['name'], item['protocol']['config'].get('temperature', 'Predeterminada')]])
    table(['Etapa', 'Repetición', 'Aciertos', 'Q', 'V'],
          [[{'known':'Conocidos', 'confirmation':'Confirmación'}[r['phase']], r['repeat']+1, f"{sum(x['success'] for x in r['rows'])}/{len(r['rows'])}",
            f"{r['q']:.1%}", f"{r['v']:.1%}"] for r in report['evaluations']])
    table(['Estado', 'Criterio completo', 'Llamadas', 'Tokens', 'Costo estimado USD'],
          [[{'completed':'Terminada', 'stopped':'Detenida'}.get(report['status'],report['status']), 'Cumple' if report.get('passes') else 'No cumple', report['calls'], report['tokens'],
            f"{report['estimated_cost_usd']:.6f}"]])
    rows = []
    for case in item['protocol']['confirmation_cases']:
        outputs = [row for r in report['evaluations'] if r['phase']=='confirmation' for row in r['rows'] if row['id']==case['id']]
        if not outputs: continue
        rows.append([case['id'], json.dumps(case['expected'],ensure_ascii=False),
                     f"{sum(r['success'] for r in outputs)}/{len(outputs)}",
                     '; '.join(sorted({json.dumps(r['raw_reply'],ensure_ascii=False) for r in outputs}))])
    if rows:
        table(['Caso reservado', 'Esperado', 'Aciertos', 'Respuestas originales'], rows)
    else:
        print('Sin confirmación: no cumplió el criterio en los casos conocidos.')

Ejecución,Temperatura
20261106T203920Z-api-reference,Predeterminada


Etapa,Repetición,Aciertos,Q,V
Conocidos,1,10/12,83.3%,8.3%


Estado,Criterio completo,Llamadas,Tokens,Costo estimado USD
Terminada,No cumple,12,8143,0.001329


Sin confirmación: no cumplió el criterio en los casos conocidos.


Ejecución,Temperatura
20261106T204025Z-api-reference,0.0


Etapa,Repetición,Aciertos,Q,V
Conocidos,1,8/12,66.7%,8.3%


Estado,Criterio completo,Llamadas,Tokens,Costo estimado USD
Terminada,No cumple,12,8143,0.001329


Sin confirmación: no cumplió el criterio en los casos conocidos.


## 6. Repetir la prueba

Desde la carpeta del caso, con Python 3.12, MLX y los pesos fijados en caché:

```sh
HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 python utils/inference_tuning.py --model qwen3-8b --thinking
```

Para repetir el barrido directo, omitir `--thinking`; `--model qwen3-32b` usa el modelo mayor. La confirmación requiere una ejecución de ajuste que cumpla el criterio y conserve fuentes, pesos, política y casos:

```sh
HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 python utils/inference_tuning.py --model qwen3-8b --confirmation-of results/RUN_NAME
```

[Script](../utils/inference_tuning.py), [casos de confirmación](../inference-confirmation-cases.yaml) y [pruebas anteriores](local-model-test.ipynb). Los nombres exactos de las ejecuciones utilizadas figuran en la primera celda.

Para repetir el control API se necesita `OPENAI_API_KEY` en el entorno:

```sh
python utils/api_reference_test.py --allow-live --temperature 0
```

[Script API](../utils/api_reference_test.py). Límite: 36 llamadas, 50.000 tokens, 900 segundos y USD 0,25 estimados. La lectura del notebook no consume API. Tarifas: USD 0,15/M de entrada y 0,60/M de salida, sin descontar caché. [Ficha oficial](https://developers.openai.com/api/docs/models/gpt-4o-mini).

GPT-3.5 Turbo y GPT-4 figuran retirados el 23 de octubre de 2026; GPT-4o mini tiene retirada prevista para el 14 de enero de 2027. Es un control temporal, no la base propuesta para la tesis. [Estado oficial](https://developers.openai.com/api/docs/deprecations).


**Siguiente paso:** fijar esta inferencia para comparar B1 y B2, con el mismo tope de tokens y contando también el razonamiento. El éxito del control manual permite retomar esa prueba; no anticipa qué mejorador funcionará mejor.
